# M45 Kaggle — 01 Build DB/index

Notebook này giữ nguyên pipeline M45. Bật **Internet** và **GPU T4**. Add một private Kaggle Dataset chứa đúng `legal-agentic-rag-m45-source.zip` và `selected-contexts.zip`, sau đó Run All. Output cuối là artifact archive dùng cho notebook 02.

In [ ]:
from pathlib import Path
import torch

INPUT_ROOT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
REPO = WORKING / 'legal-agentic-rag'
CONFIG_TEMPLATE = REPO / 'configs/uit-dsc-2026-task2-m45-qwen3-colab.example.json'
CONFIG = WORKING / 'm45-kaggle-config.json'
ARTIFACTS = WORKING / 'uit-dsc-2026-task2-m45-artifacts'
ARTIFACT_ARCHIVE = WORKING / 'uit-dsc-2026-task2-m45-artifacts.tar.gz'
ARTIFACT_CHECKSUM = WORKING / 'uit-dsc-2026-task2-m45-artifacts.tar.gz.sha256'

def optional_unique_input(filename: str) -> Path | None:
    matches = list(INPUT_ROOT.rglob(filename))
    assert len(matches) <= 1, f'Có nhiều file {filename}: {matches}'
    return matches[0] if matches else None

SOURCE_ZIP = optional_unique_input('legal-agentic-rag-m45-source.zip')
source_directories = [
    path.parent for path in INPUT_ROOT.rglob('pyproject.toml')
    if (path.parent / 'src/legal_agentic_rag').is_dir()
    and (path.parent / 'configs/uit-dsc-2026-task2-m45-qwen3-colab.example.json').is_file()
]
assert SOURCE_ZIP is not None or len(source_directories) >= 1, (
    f'Không thấy source ZIP hoặc source đã giải nén. Input roots: {list(INPUT_ROOT.iterdir())}'
)
SOURCE_DIRECTORY = sorted(source_directories)[0] if SOURCE_ZIP is None else None
CONTEXT_ZIP = optional_unique_input('selected-contexts.zip')
if CONTEXT_ZIP is not None:
    CONTEXTS = CONTEXT_ZIP
else:
    context_files = list(INPUT_ROOT.rglob('context_*.json'))
    context_directories = {path.parent for path in context_files}
    assert len(context_files) == 8532 and len(context_directories) == 1, (
        f'Không thấy canonical context ZIP/directory: files={len(context_files)}, dirs={context_directories}'
    )
    CONTEXTS = next(iter(context_directories))
assert torch.cuda.is_available(), 'Hãy bật GPU Accelerator trong Kaggle Settings'
torch.ones(1, device='cuda')
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print('Source:', SOURCE_ZIP or SOURCE_DIRECTORY)
print('Contexts:', CONTEXTS)

In [ ]:
from zipfile import ZipFile
import shutil

if not REPO.is_dir():
    if SOURCE_ZIP is not None:
        with ZipFile(SOURCE_ZIP) as archive:
            archive.extractall(WORKING)
    else:
        shutil.copytree(SOURCE_DIRECTORY, REPO)
assert CONFIG_TEMPLATE.is_file(), CONFIG_TEMPLATE
print('Repository:', REPO)

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
    'transformers==5.15.0', 'sentence-transformers==5.4.1'
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), '--no-deps'], check=True)
repo_src = str(REPO / 'src')
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)

import legal_agentic_rag
import sentence_transformers
import transformers
print('Project:', legal_agentic_rag.__version__)
print('Sentence Transformers:', sentence_transformers.__version__)
print('Transformers:', transformers.__version__)
assert legal_agentic_rag.__version__ == '0.45.0'
assert sentence_transformers.__version__ == '5.4.1'
assert transformers.__version__ == '5.15.0'

In [ ]:
from hashlib import sha256
import json

def file_sha256(path: Path) -> str:
    digest = sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

if CONTEXTS.is_file():
    assert file_sha256(CONTEXTS) == 'ebcfc896df06087e7da532b4653f32adfaba2200c8ed92a0069e46dbfa126a97'
else:
    from legal_agentic_rag.competition.uit_dsc_2026.loader import UitDsc2026DataLoader
    identity = UitDsc2026DataLoader().inspect_context_source(CONTEXTS)
    print('Extracted context revision:', identity.revision)
    print('Extracted context records:', identity.member_count)
    assert identity.revision == 'sha256:9a4441b4537ceb646b15359f470a1da0904e6c92a61e8c4c376c19e17dec395e'
    assert identity.member_count == 8532
payload = json.loads(CONFIG_TEMPLATE.read_text(encoding='utf-8'))
payload['artifacts']['root_path'] = str(ARTIFACTS)
CONFIG.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Config:', CONFIG)
print('Giữ nguyên models:', payload['offline']['embedding']['model_name'], payload['online']['reranker']['model_name'], payload['online']['generation']['model_name'])
print('Giữ nguyên candidate/top/context/output:', payload['online']['retrieval']['candidate_k'], payload['online']['retrieval']['top_k'], payload['online']['generation']['max_context_tokens'], payload['online']['generation']['max_output_tokens'])

## Full build
Đây là cell lâu nhất. Không chạy hai build song song. Durable stages/vector checkpoints cho phép chạy lại cell trong cùng session nếu bị dừng mềm.

In [ ]:
subprocess.run([
    'legal-rag-build-competition', '--config', str(CONFIG), '--source', str(CONTEXTS)
], cwd=REPO, check=True)

In [ ]:
serving_metadata = ARTIFACTS / 'vector_serving/metadata.sqlite3'
if not serving_metadata.is_file():
    subprocess.run(['legal-rag-prepare-serving', '--config', str(CONFIG)], cwd=REPO, check=True)
report = json.loads((ARTIFACTS / 'build_validation_full_corpus.json').read_text(encoding='utf-8'))
required = [
    'dataset_manifest.json', 'audit/corpus_audit.json', 'legal_chunks/manifest.json',
    'bm25/index.sqlite3', 'bm25/manifest.json', 'vector/vectors.npy',
    'vector/chunks.jsonl', 'vector/manifest.json',
    'vector_serving/metadata.sqlite3', 'vector_serving/manifest.json',
    'build_validation_full_corpus.json',
]
for relative in required:
    path = ARTIFACTS / relative
    print('OK' if path.is_file() else 'MISSING', relative)
    assert path.is_file(), path
assert report['is_valid'] is True
print('DB/index M45 hợp lệ.')

## Đóng gói artifact làm Kaggle Dataset
Sau khi cell hoàn tất, chọn **Save Version → Save & Run All**, rồi dùng output archive/checksum của notebook này để tạo private Kaggle Dataset hoặc add notebook output trực tiếp vào notebook 02.

In [ ]:
if not ARTIFACT_ARCHIVE.is_file():
    subprocess.run([
        'tar', '-czf', str(ARTIFACT_ARCHIVE), '-C', str(WORKING), ARTIFACTS.name
    ], check=True)
archive_hash = file_sha256(ARTIFACT_ARCHIVE)
ARTIFACT_CHECKSUM.write_text(f'{archive_hash}  {ARTIFACT_ARCHIVE.name}\n', encoding='utf-8')
print('Artifact:', ARTIFACT_ARCHIVE)
print('Size GiB:', round(ARTIFACT_ARCHIVE.stat().st_size / 2**30, 3))
print('SHA-256:', archive_hash)
print('Checksum:', ARTIFACT_CHECKSUM)